# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset ([Kulka, M. et al., 2026](https://sen.science/doi/10.71728/senscience.vq4a-28xa)) using the `mlcroissant` library. The notebook will guide you through loading the Croissant schema, inspecting available record sets and fields, and performing exploratory analysis programmatically referencing entity `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`. Start by loading dataset metadata and exploring core metadata fields for context.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View metadata information
print("Dataset name:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Published:", dataset.metadata.datePublished)
print("Version:", dataset.metadata.version)
print("License:", dataset.metadata.license)
print("Cite as:", dataset.metadata.citeAs)
print("Keywords:", dataset.metadata.keywords)

# Show data collection and preprocessing notes
print("Data Collection:", getattr(dataset.metadata, 'dataCollection', None))
print("Data Preprocessing Protocol:", getattr(dataset.metadata, 'dataPreprocessingProtocol', None))

## 2. Data Overview
Review available record sets, referencing their `@id` and inspect their fields with all field `@id`s. 
This helps determine what data tables (record sets) the dataset contains and how to address each field uniquely for further analysis.

> In mlcroissant, use `dataset.metadata.recordSets` and inspect each `recordSet`'s `.id` field, as well as its fields.

In [ ]:
# List the available Record Sets and their fields by @id:
record_sets = getattr(dataset.metadata, 'recordSets', [])
if not record_sets:
    # Support for single recordSet (for some croissant files):
    single_record_set = getattr(dataset.metadata, 'recordSet', [])
    if isinstance(single_record_set, list):
        record_sets = single_record_set
    elif single_record_set:
        record_sets = [single_record_set]

print("Available Record Sets and their fields (with @id):\n")
record_set_ids = []
record_set_field_ids = dict()
for rs in record_sets:
    print(f"  RecordSet name: {getattr(rs, 'name', 'N/A')}")
    print(f"    @id: {getattr(rs, 'id', None)}")
    record_set_ids.append(getattr(rs, 'id', None))
    fields = getattr(rs, 'fields', [])
    field_ids = []
    print("    Available fields:")
    for f in fields:
        f_id = getattr(f, 'id', None)
        field_ids.append(f_id)
        print(f"      - {getattr(f, 'name', f_id)} (@id: {f_id}) type: {getattr(f, 'dataType', 'N/A')}")
    record_set_field_ids[getattr(rs, 'id', None)] = field_ids
    print('')

# If no record sets found, print a warning
if not record_set_ids:
    print("No record sets found in this dataset's Croissant schema.")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. You must reference the record set by `@id`. Also, print the columns (field `@id`s), and view a sample of data for each record set.

This setup prepares named DataFrames for each record set, using their `@id` as the dictionary key.

In [ ]:
# Extract data from every available record set (by @id)
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"--- Columns (@id): {list(df.columns)}")
    display(df.head())  # Show a preview of the DataFrame
    print("\n")

# If you know the main record set of interest (e.g. protein measurement table), assign it below for quick reference
main_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Common data processing steps: filter records based on criteria, normalize numeric fields, and group by protein attribute or other key attributes.

Replace the `<numeric_field_id>` and `<group_field_id>` with actual `@id` strings from the record set fields obtained above.

In [ ]:
from pandas.api.types import is_numeric_dtype

# You may have to update with the correct @id. List all columns for reference:
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Try to automatically find a numeric field and a group field
    numeric_field_id = None
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    group_field_id = None
    # Try to use a string field as group field
    for col in df.columns:
        if df[col].dtype == object:
            group_field_id = col
            break

    print(f"Using numeric field: {numeric_field_id}")
    print(f"Using group field: {group_field_id}")

    # Filtering: keep records where numeric field > threshold
    threshold = df[numeric_field_id].mean() if numeric_field_id else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f'Filtered records with {numeric_field_id} > {threshold} (mean):')
    display(filtered_df.head())

    # Normalize the chosen numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f'Normalized {numeric_field_id} for filtered records:')
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group field and show means
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}, showing average {numeric_field_id} per group:")
        display(grouped_df.head())
else:
    print("No main record set available for EDA.")

## 5. Visualization
Visualize distributions or relationships for a numeric column. For example, plot the histogram for a numeric field and a barplot for mean values per group.

In [ ]:
import matplotlib.pyplot as plt

if main_record_set_id and main_record_set_id in dataframes and numeric_field_id:
    plt.figure(figsize=(7, 4))
    df[numeric_field_id].hist(bins=30)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    # If group_field_id is set, show grouped bar plot
    if group_field_id:
        # Show only top-20 groups by count
        mean_values = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(20)
        plt.figure(figsize=(12, 5))
        mean_values.plot(kind='bar')
        plt.ylabel(f'Mean of {numeric_field_id}')
        plt.title(f'Mean {numeric_field_id} per {group_field_id} (top 20)')
        plt.show()


## 6. Conclusion
We have successfully explored the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset programmatically using the `mlcroissant` library:
- Inspected dataset metadata and available record sets and fields (referenced by their Croissant `@id`).
- Loaded data into pandas DataFrames and viewed the underlying structure via record sets.
- Performed exploratory analysis, including filtering and normalization, on available numeric fields.
- Visualized distributions and group-level means.

Continue with further custom analysis or machine learning as needed for your research. For more, see [mlcroissant documentation](https://mlcommons.github.io/croissant/python/).